In [1]:
import math
import random
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms


SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


BATCH_SIZE = 128
EPOCHS = 20
LR = 1e-3
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 1.0

FEATURE_DIM = 128
NUM_CLASSES = 10

RECKAN_ORDER = 5
CNN_HIDDEN_DIM = 55

RECURRENCE_LR_MULTIPLIER = 0.25
COEFF_BOUND = 2.5

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,))
])

train_dataset = datasets.FashionMNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.FashionMNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)


class CNNBackbone(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout(0.10),

            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout(0.15),

            nn.Conv2d(64, FEATURE_DIM, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(FEATURE_DIM),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d((1, 1))
        )

    def forward(self, x):
        return self.features(x).flatten(1)


class MLPClassifier(nn.Module):
    def __init__(self, in_features=FEATURE_DIM,
                 hidden_features=CNN_HIDDEN_DIM,
                 num_classes=NUM_CLASSES):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(in_features, hidden_features),
            nn.ReLU(inplace=True),
            nn.Dropout(0.20),
            nn.Linear(hidden_features, num_classes)
        )

    def forward(self, x):
        return self.net(x)

class StandardCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = CNNBackbone()
        self.classifier = MLPClassifier()

    def forward(self, x):
        return self.classifier(self.backbone(x))

class RecKANClassifier(nn.Module):
    def __init__(self, in_features=FEATURE_DIM,
                 num_classes=NUM_CLASSES,
                 order=RECKAN_ORDER,
                 coeff_bound=COEFF_BOUND):
        super().__init__()

        self.in_features = in_features
        self.num_classes = num_classes
        self.order = order
        self.coeff_bound = coeff_bound

        def inverse_tanh(value):
            normalized = max(min(value / coeff_bound, 0.999), -0.999)
            return 0.5 * math.log((1.0 + normalized) / (1.0 - normalized))

        # Start at the exact Chebyshev-U recurrence:
        # R_{n+1}=2x R_n - R_{n-1}, R_0=0, R_1=1.
        self.raw_a = nn.Parameter(torch.tensor(inverse_tanh(0.0)))
        self.raw_b = nn.Parameter(torch.tensor(inverse_tanh(2.0)))
        self.raw_c = nn.Parameter(torch.tensor(inverse_tanh(0.0)))
        self.raw_d = nn.Parameter(torch.tensor(inverse_tanh(0.0)))
        self.raw_e = nn.Parameter(torch.tensor(inverse_tanh(-1.0)))

        self.weight = nn.Parameter(
            torch.empty(in_features, order + 1, num_classes)
        )
        self.bias = nn.Parameter(torch.zeros(num_classes))

        nn.init.xavier_uniform_(self.weight)

    def recurrence_coefficients(self):
        a = self.coeff_bound * torch.tanh(self.raw_a)
        b = self.coeff_bound * torch.tanh(self.raw_b)
        c = self.coeff_bound * torch.tanh(self.raw_c)
        d = self.coeff_bound * torch.tanh(self.raw_d)
        e = self.coeff_bound * torch.tanh(self.raw_e)
        return a, b, c, d, e

    def forward(self, features):
        x = torch.tanh(features)

        a, b, c, d, e = self.recurrence_coefficients()

        r0 = torch.zeros_like(x)
        r1 = torch.ones_like(x)
        basis = [r0, r1]

        for _ in range(1, self.order):
            r_next = (
                (a * x.square() + b * x + c) * basis[-1]
                + (d * x + e) * basis[-2]
            )

            scale = r_next.detach().abs().amax().clamp_min(1e-6)
            r_next = r_next / scale
            basis.append(r_next)

        basis = torch.stack(basis, dim=-1)

        return torch.einsum("bik,ikc->bc", basis, self.weight) + self.bias

class CNNRecKAN(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = CNNBackbone()
        self.classifier = RecKANClassifier()

    def forward(self, x):
        return self.classifier(self.backbone(x))

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def count_classifier_parameters(model):
    return sum(
        p.numel() for p in model.classifier.parameters()
        if p.requires_grad
    )

@torch.no_grad()
def evaluate(model, loader):
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        logits = model(images)
        loss = F.cross_entropy(logits, labels)

        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, 100.0 * correct / total

def train_one_epoch(model, loader, optimizer):
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        logits = model(images)
        loss = F.cross_entropy(logits, labels)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, 100.0 * correct / total

def make_optimizer(model, use_reckan=False):
    if not use_reckan:
        return torch.optim.AdamW(
            model.parameters(),
            lr=LR,
            weight_decay=WEIGHT_DECAY
        )

    recurrence_parameters = [
        model.classifier.raw_a,
        model.classifier.raw_b,
        model.classifier.raw_c,
        model.classifier.raw_d,
        model.classifier.raw_e
    ]

    recurrence_ids = {id(p) for p in recurrence_parameters}

    regular_parameters = [
        p for p in model.parameters()
        if id(p) not in recurrence_ids
    ]

    return torch.optim.AdamW(
        [
            {"params": regular_parameters, "lr": LR},
            {
                "params": recurrence_parameters,
                "lr": LR * RECURRENCE_LR_MULTIPLIER
            }
        ],
        weight_decay=WEIGHT_DECAY
    )

def train_model(model, name, use_reckan=False):
    model = model.to(device)
    optimizer = make_optimizer(model, use_reckan=use_reckan)

    best_accuracy = -1.0
    best_epoch = 0
    best_state = None
    history = []

    print("\n" + "=" * 82)
    print(name)
    print(f"Backbone parameters:   {count_parameters(model.backbone):,}")
    print(f"Classifier parameters: {count_classifier_parameters(model):,}")
    print(f"Total parameters:      {count_parameters(model):,}")
    print("=" * 82)

    for epoch in range(1, EPOCHS + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer)
        test_loss, test_acc = evaluate(model, test_loader)

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_accuracy": train_acc,
            "test_loss": test_loss,
            "test_accuracy": test_acc
        })

        if test_acc > best_accuracy:
            best_accuracy = test_acc
            best_epoch = epoch
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }

        print(
            f"Epoch {epoch:02d}/{EPOCHS} | "
            f"Train: {train_acc:6.2f}% | "
            f"Test: {test_acc:6.2f}% | "
            f"Test loss: {test_loss:.4f}"
        )

    model.load_state_dict(best_state)

    result = {
        "name": name,
        "model": model,
        "history": history,
        "backbone_parameters": count_parameters(model.backbone),
        "classifier_parameters": count_classifier_parameters(model),
        "total_parameters": count_parameters(model),
        "best_test_accuracy": best_accuracy,
        "best_epoch": best_epoch
    }

    if use_reckan:
        a, b, c, d, e = model.classifier.recurrence_coefficients()

        result["learned_recurrence"] = {
            "a": a.item(),
            "b": b.item(),
            "c": c.item(),
            "d": d.item(),
            "e": e.item()
        }

    return result


torch.manual_seed(SEED)
standard_cnn = StandardCNN()

torch.manual_seed(SEED)
cnn_reckan = CNNRecKAN()

standard_classifier_params = count_classifier_parameters(standard_cnn)
reckan_classifier_params = count_classifier_parameters(cnn_reckan)

relative_gap = (
    abs(standard_classifier_params - reckan_classifier_params)
    / reckan_classifier_params
) * 100.0

print("\nParameter matching check")
print(f"Standard CNN classifier: {standard_classifier_params:,}")
print(f"CNN-RecKAN classifier:   {reckan_classifier_params:,}")
print(f"Classifier difference:    {abs(standard_classifier_params - reckan_classifier_params):,}")
print(f"Relative difference:      {relative_gap:.3f}%")

# ============================================================
# 9. Train both models
# ============================================================
torch.manual_seed(SEED)
standard_result = train_model(
    standard_cnn,
    name="Standard CNN (matched MLP classifier)",
    use_reckan=False
)

torch.manual_seed(SEED)
reckan_result = train_model(
    cnn_reckan,
    name="CNN-RecKAN (matched RecKAN classifier)",
    use_reckan=True
)

# ============================================================
# 10. Report results
# ============================================================
print("\n" + "=" * 82)
print("FINAL RESULTS: FASHION-MNIST")
print("=" * 82)

print(
    f"{'Model':<42}"
    f"{'Total params':>15}"
    f"{'Best test acc.':>18}"
    f"{'Epoch':>8}"
)
print("-" * 82)

for result in [standard_result, reckan_result]:
    print(
        f"{result['name']:<42}"
        f"{result['total_parameters']:>15,}"
        f"{result['best_test_accuracy']:>17.2f}%"
        f"{result['best_epoch']:>8}"
    )

accuracy_difference = (
    reckan_result["best_test_accuracy"]
    - standard_result["best_test_accuracy"]
)

print("-" * 82)
print(f"CNN-RecKAN minus Standard CNN: {accuracy_difference:+.2f} percentage points")

print("\nLearned RecKAN recurrence coefficients:")
for key, value in reckan_result["learned_recurrence"].items():
    print(f"  {key} = {value:+.4f}")

Device: cuda


100%|██████████| 26.4M/26.4M [00:00<00:00, 112MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 4.23MB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 52.0MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 9.88MB/s]



Parameter matching check
Standard CNN classifier: 7,655
CNN-RecKAN classifier:   7,695
Classifier difference:    40
Relative difference:      0.520%

Standard CNN (matched MLP classifier)
Backbone parameters:   139,168
Classifier parameters: 7,655
Total parameters:      146,823
Epoch 01/20 | Train:  77.42% | Test:  84.51% | Test loss: 0.4339
Epoch 02/20 | Train:  87.22% | Test:  86.34% | Test loss: 0.3901
Epoch 03/20 | Train:  89.29% | Test:  89.61% | Test loss: 0.2896
Epoch 04/20 | Train:  90.15% | Test:  90.28% | Test loss: 0.2702
Epoch 05/20 | Train:  91.15% | Test:  90.69% | Test loss: 0.2560
Epoch 06/20 | Train:  91.56% | Test:  91.59% | Test loss: 0.2344
Epoch 07/20 | Train:  92.02% | Test:  91.92% | Test loss: 0.2288
Epoch 08/20 | Train:  92.30% | Test:  90.55% | Test loss: 0.2560
Epoch 09/20 | Train:  92.69% | Test:  91.02% | Test loss: 0.2534
Epoch 10/20 | Train:  93.10% | Test:  92.20% | Test loss: 0.2114
Epoch 11/20 | Train:  93.28% | Test:  92.03% | Test loss: 0.2353
Epoch

In [2]:
import math
import random
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# ============================================================
# 1. Reproducibility and device
# ============================================================
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ============================================================
# 2. Experiment configuration
# ============================================================
BATCH_SIZE = 128
TEST_BATCH_SIZE = 256
EPOCHS = 50
LR = 1e-3
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 1.0

FEATURE_DIM = 128
NUM_CLASSES = 10

RECKAN_ORDER = 5
CNN_HIDDEN_DIM = 55

RECURRENCE_LR_MULTIPLIER = 0.25
COEFF_BOUND = 2.5

# ============================================================
# 3. CIFAR-10 data
#
# Train augmentation is used identically for both models.
# Test data are normalized only.
# ============================================================
cifar_mean = (0.4914, 0.4822, 0.4465)
cifar_std = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(cifar_mean, cifar_std),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(cifar_mean, cifar_std),
])

train_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=train_transform,
)

test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=test_transform,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=TEST_BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=True,
)

class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
]

# ============================================================
# 4. Identical CIFAR-10 CNN backbone
#
# The output is a 128-dimensional feature vector for both models.
# ============================================================
class CIFARBackbone(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),              # 32x32 -> 16x16
            nn.Dropout(0.10),

            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),              # 16x16 -> 8x8
            nn.Dropout(0.15),

            nn.Conv2d(64, FEATURE_DIM, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(FEATURE_DIM),
            nn.ReLU(inplace=True),

            nn.Conv2d(FEATURE_DIM, FEATURE_DIM, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(FEATURE_DIM),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),              # 8x8 -> 4x4
            nn.Dropout(0.20),

            nn.AdaptiveAvgPool2d((1, 1)),
        )

    def forward(self, x):
        return self.features(x).flatten(1)

# ============================================================
# 5. Matched standard CNN classifier
#
# 128 -> 55 -> 10:
# 128*55 + 55 + 55*10 + 10 = 7,655 parameters.
# ============================================================
class MLPClassifier(nn.Module):
    def __init__(
        self,
        in_features=FEATURE_DIM,
        hidden_features=CNN_HIDDEN_DIM,
        num_classes=NUM_CLASSES,
    ):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(in_features, hidden_features),
            nn.ReLU(inplace=True),
            nn.Dropout(0.20),
            nn.Linear(hidden_features, num_classes),
        )

    def forward(self, x):
        return self.net(x)

class StandardCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = CIFARBackbone()
        self.classifier = MLPClassifier()

    def forward(self, x):
        return self.classifier(self.backbone(x))

# ============================================================
# 6. RecKAN classifier
#
# R_0(x)=0, R_1(x)=1
# R_{n+1}(x)=(a x^2+b x+c)R_n(x)+(d x+e)R_{n-1}(x)
#
# Classifier parameters:
# 128*(5+1)*10 + 10 + 5 = 7,695.
# ============================================================
class RecKANClassifier(nn.Module):
    def __init__(
        self,
        in_features=FEATURE_DIM,
        num_classes=NUM_CLASSES,
        order=RECKAN_ORDER,
        coeff_bound=COEFF_BOUND,
    ):
        super().__init__()

        self.in_features = in_features
        self.num_classes = num_classes
        self.order = order
        self.coeff_bound = coeff_bound

        def inverse_tanh(value):
            normalized = max(min(value / coeff_bound, 0.999), -0.999)
            return 0.5 * math.log((1.0 + normalized) / (1.0 - normalized))

        # Exact Chebyshev-U initialization:
        # R_{n+1}(x)=2xR_n(x)-R_{n-1}(x),
        # R_0=0 and R_1=1.
        self.raw_a = nn.Parameter(torch.tensor(inverse_tanh(0.0)))
        self.raw_b = nn.Parameter(torch.tensor(inverse_tanh(2.0)))
        self.raw_c = nn.Parameter(torch.tensor(inverse_tanh(0.0)))
        self.raw_d = nn.Parameter(torch.tensor(inverse_tanh(0.0)))
        self.raw_e = nn.Parameter(torch.tensor(inverse_tanh(-1.0)))

        self.weight = nn.Parameter(
            torch.empty(in_features, order + 1, num_classes)
        )
        self.bias = nn.Parameter(torch.zeros(num_classes))

        nn.init.xavier_uniform_(self.weight)

    def recurrence_coefficients(self):
        a = self.coeff_bound * torch.tanh(self.raw_a)
        b = self.coeff_bound * torch.tanh(self.raw_b)
        c = self.coeff_bound * torch.tanh(self.raw_c)
        d = self.coeff_bound * torch.tanh(self.raw_d)
        e = self.coeff_bound * torch.tanh(self.raw_e)
        return a, b, c, d, e

    def forward(self, features):
        x = torch.tanh(features)

        a, b, c, d, e = self.recurrence_coefficients()

        r0 = torch.zeros_like(x)
        r1 = torch.ones_like(x)
        basis = [r0, r1]

        for _ in range(1, self.order):
            r_next = (
                (a * x.square() + b * x + c) * basis[-1]
                + (d * x + e) * basis[-2]
            )

            # One detached batch-wide scale per recurrence step.
            scale = r_next.detach().abs().amax().clamp_min(1e-6)
            r_next = r_next / scale
            basis.append(r_next)

        basis = torch.stack(basis, dim=-1)

        return torch.einsum("bik,ikc->bc", basis, self.weight) + self.bias

class CNNRecKAN(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = CIFARBackbone()
        self.classifier = RecKANClassifier()

    def forward(self, x):
        return self.classifier(self.backbone(x))

# ============================================================
# 7. Parameter counting and training utilities
# ============================================================
def count_parameters(module):
    return sum(p.numel() for p in module.parameters() if p.requires_grad)

def count_classifier_parameters(model):
    return count_parameters(model.classifier)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        logits = model(images)
        loss = F.cross_entropy(logits, labels)

        total_loss += loss.item() * labels.size(0)
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_examples += labels.size(0)

    return total_loss / total_examples, 100.0 * total_correct / total_examples

def train_one_epoch(model, loader, optimizer):
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        logits = model(images)
        loss = F.cross_entropy(logits, labels)
        loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)

        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_examples += labels.size(0)

    return total_loss / total_examples, 100.0 * total_correct / total_examples

def make_optimizer(model, use_reckan=False):
    if not use_reckan:
        return torch.optim.AdamW(
            model.parameters(),
            lr=LR,
            weight_decay=WEIGHT_DECAY,
        )

    recurrence_parameters = [
        model.classifier.raw_a,
        model.classifier.raw_b,
        model.classifier.raw_c,
        model.classifier.raw_d,
        model.classifier.raw_e,
    ]

    recurrence_ids = {id(p) for p in recurrence_parameters}
    regular_parameters = [
        p for p in model.parameters()
        if id(p) not in recurrence_ids
    ]

    return torch.optim.AdamW(
        [
            {"params": regular_parameters, "lr": LR},
            {
                "params": recurrence_parameters,
                "lr": LR * RECURRENCE_LR_MULTIPLIER,
            },
        ],
        weight_decay=WEIGHT_DECAY,
    )

def train_model(model, name, use_reckan=False):
    model = model.to(device)
    optimizer = make_optimizer(model, use_reckan=use_reckan)

    best_accuracy = -1.0
    best_epoch = 0
    best_state = None
    history = []

    print("\n" + "=" * 86)
    print(name)
    print(f"Backbone parameters:   {count_parameters(model.backbone):,}")
    print(f"Classifier parameters: {count_classifier_parameters(model):,}")
    print(f"Total parameters:      {count_parameters(model):,}")
    print("=" * 86)

    for epoch in range(1, EPOCHS + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer)
        test_loss, test_acc = evaluate(model, test_loader)

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_accuracy": train_acc,
            "test_loss": test_loss,
            "test_accuracy": test_acc,
        })

        if test_acc > best_accuracy:
            best_accuracy = test_acc
            best_epoch = epoch
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }

        print(
            f"Epoch {epoch:02d}/{EPOCHS} | "
            f"Train: {train_acc:6.2f}% | "
            f"Test: {test_acc:6.2f}% | "
            f"Test loss: {test_loss:.4f}"
        )

    model.load_state_dict(best_state)

    result = {
        "name": name,
        "backbone_parameters": count_parameters(model.backbone),
        "classifier_parameters": count_classifier_parameters(model),
        "total_parameters": count_parameters(model),
        "best_test_accuracy": best_accuracy,
        "best_epoch": best_epoch,
        "history": history,
    }

    if use_reckan:
        a, b, c, d, e = model.classifier.recurrence_coefficients()
        result["recurrence"] = {
            "a": a.item(),
            "b": b.item(),
            "c": c.item(),
            "d": d.item(),
            "e": e.item(),
        }

    return result

# ============================================================
# 8. Build models and verify close parameter matching
# ============================================================
torch.manual_seed(SEED)
standard_cnn = StandardCNN()

torch.manual_seed(SEED)
cnn_reckan = CNNRecKAN()

standard_total = count_parameters(standard_cnn)
reckan_total = count_parameters(cnn_reckan)
standard_head = count_classifier_parameters(standard_cnn)
reckan_head = count_classifier_parameters(cnn_reckan)

print("\n" + "=" * 86)
print("PARAMETER-MATCHING CHECK")
print("=" * 86)
print(f"Standard CNN classifier parameters: {standard_head:,}")
print(f"CNN-RecKAN classifier parameters:   {reckan_head:,}")
print(f"Classifier difference:               {abs(reckan_head - standard_head):,}")
print(f"Standard CNN total parameters:       {standard_total:,}")
print(f"CNN-RecKAN total parameters:         {reckan_total:,}")
print(f"Total difference:                    {abs(reckan_total - standard_total):,}")
print(
    f"Relative total difference:           "
    f"{100.0 * abs(reckan_total - standard_total) / reckan_total:.4f}%"
)

# ============================================================
# 9. Train the matched models
# ============================================================
torch.manual_seed(SEED)
standard_result = train_model(
    standard_cnn,
    name="Standard CNN (matched MLP classifier)",
    use_reckan=False,
)

torch.manual_seed(SEED)
reckan_result = train_model(
    cnn_reckan,
    name="CNN-RecKAN (matched RecKAN classifier)",
    use_reckan=True,
)

# ============================================================
# 10. Final report
# ============================================================
print("\n" + "=" * 86)
print("FINAL RESULTS: CIFAR-10")
print("=" * 86)
print(
    f"{'Model':<44}"
    f"{'Total params':>15}"
    f"{'Best test acc.':>18}"
    f"{'Epoch':>8}"
)
print("-" * 86)

for result in [standard_result, reckan_result]:
    print(
        f"{result['name']:<44}"
        f"{result['total_parameters']:>15,}"
        f"{result['best_test_accuracy']:>17.2f}%"
        f"{result['best_epoch']:>8}"
    )

difference = (
    reckan_result["best_test_accuracy"]
    - standard_result["best_test_accuracy"]
)

print("-" * 86)
print(f"CNN-RecKAN minus Standard CNN: {difference:+.2f} percentage points")

print("\nLearned RecKAN recurrence coefficients:")
for name, value in reckan_result["recurrence"].items():
    print(f"  {name} = {value:+.4f}")

Device: cuda


100%|██████████| 170M/170M [33:30<00:00, 84.8kB/s] 



PARAMETER-MATCHING CHECK
Standard CNN classifier parameters: 7,655
CNN-RecKAN classifier parameters:   7,695
Classifier difference:               40
Standard CNN total parameters:       295,111
CNN-RecKAN total parameters:         295,151
Total difference:                    40
Relative total difference:           0.0136%

Standard CNN (matched MLP classifier)
Backbone parameters:   287,456
Classifier parameters: 7,655
Total parameters:      295,111
Epoch 01/50 | Train:  44.29% | Test:  56.48% | Test loss: 1.2157
Epoch 02/50 | Train:  61.21% | Test:  61.53% | Test loss: 1.1766
Epoch 03/50 | Train:  66.57% | Test:  69.87% | Test loss: 0.8578
Epoch 04/50 | Train:  70.64% | Test:  69.52% | Test loss: 0.9200
Epoch 05/50 | Train:  73.35% | Test:  74.42% | Test loss: 0.7451
Epoch 06/50 | Train:  75.47% | Test:  75.22% | Test loss: 0.7284
Epoch 07/50 | Train:  77.21% | Test:  79.05% | Test loss: 0.6215
Epoch 08/50 | Train:  78.37% | Test:  78.59% | Test loss: 0.6249
Epoch 09/50 | Train:  79.

In [3]:
import math
import random
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# ============================================================
# 1. Reproducibility and device
# ============================================================
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ============================================================
# 2. Experiment configuration
# ============================================================
BATCH_SIZE = 128
TEST_BATCH_SIZE = 256
EPOCHS = 50

LR = 1e-3
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 1.0

FEATURE_DIM = 128
NUM_CLASSES = 10

RECKAN_ORDER = 5
CNN_HIDDEN_DIM = 55

RECURRENCE_LR_MULTIPLIER = 0.25
COEFF_BOUND = 2.5

# ============================================================
# 3. SVHN data
#
# NOTE:
# torchvision.datasets.SVHN represents the digit "0" as label 0,
# so no label conversion is needed.
# ============================================================
svhn_mean = (0.4377, 0.4438, 0.4728)
svhn_std = (0.1980, 0.2010, 0.1970)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(svhn_mean, svhn_std),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(svhn_mean, svhn_std),
])

train_dataset = datasets.SVHN(
    root="./data",
    split="train",
    download=True,
    transform=train_transform,
)

test_dataset = datasets.SVHN(
    root="./data",
    split="test",
    download=True,
    transform=test_transform,
)

loader_kwargs = {
    "num_workers": 2,
    "pin_memory": torch.cuda.is_available(),
}

# For notebooks on Windows, replace num_workers=2 with 0 and
# do not add persistent_workers=True.
if loader_kwargs["num_workers"] > 0:
    loader_kwargs["persistent_workers"] = True

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    **loader_kwargs,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=TEST_BATCH_SIZE,
    shuffle=False,
    **loader_kwargs,
)

# ============================================================
# 4. Shared CNN backbone
#
# Both models use exactly this feature extractor.
# Output: 128-dimensional feature vector.
# ============================================================
class SVHNBackbone(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),              # 32 -> 16
            nn.Dropout(0.10),

            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),              # 16 -> 8
            nn.Dropout(0.15),

            nn.Conv2d(64, FEATURE_DIM, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(FEATURE_DIM),
            nn.ReLU(inplace=True),

            nn.Conv2d(FEATURE_DIM, FEATURE_DIM, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(FEATURE_DIM),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),              # 8 -> 4
            nn.Dropout(0.20),

            nn.AdaptiveAvgPool2d((1, 1)),
        )

    def forward(self, x):
        return self.features(x).flatten(1)

# ============================================================
# 5. Matched standard classifier
#
# 128 -> 55 -> 10:
# 128*55 + 55 + 55*10 + 10 = 7,655 trainable parameters.
# ============================================================
class MLPClassifier(nn.Module):
    def __init__(
        self,
        in_features=FEATURE_DIM,
        hidden_features=CNN_HIDDEN_DIM,
        num_classes=NUM_CLASSES,
    ):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(in_features, hidden_features),
            nn.ReLU(inplace=True),
            nn.Dropout(0.20),
            nn.Linear(hidden_features, num_classes),
        )

    def forward(self, x):
        return self.net(x)

class StandardCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = SVHNBackbone()
        self.classifier = MLPClassifier()

    def forward(self, x):
        return self.classifier(self.backbone(x))

# ============================================================
# 6. RecKAN classifier
#
# R_0=0, R_1=1,
# R_{n+1}=(a*x^2+b*x+c)R_n+(d*x+e)R_{n-1}.
#
# Classifier count:
# 128*(5+1)*10 + 10 + 5 = 7,695.
# ============================================================
class RecKANClassifier(nn.Module):
    def __init__(
        self,
        in_features=FEATURE_DIM,
        num_classes=NUM_CLASSES,
        order=RECKAN_ORDER,
        coeff_bound=COEFF_BOUND,
    ):
        super().__init__()

        self.order = order
        self.coeff_bound = coeff_bound

        def inverse_tanh(value):
            normalized = max(min(value / coeff_bound, 0.999), -0.999)
            return 0.5 * math.log((1.0 + normalized) / (1.0 - normalized))

        # Exact Chebyshev-U initialization:
        # R_{n+1}=2xR_n-R_{n-1}.
        self.raw_a = nn.Parameter(torch.tensor(inverse_tanh(0.0)))
        self.raw_b = nn.Parameter(torch.tensor(inverse_tanh(2.0)))
        self.raw_c = nn.Parameter(torch.tensor(inverse_tanh(0.0)))
        self.raw_d = nn.Parameter(torch.tensor(inverse_tanh(0.0)))
        self.raw_e = nn.Parameter(torch.tensor(inverse_tanh(-1.0)))

        self.weight = nn.Parameter(
            torch.empty(in_features, order + 1, num_classes)
        )
        self.bias = nn.Parameter(torch.zeros(num_classes))

        nn.init.xavier_uniform_(self.weight)

    def recurrence_coefficients(self):
        a = self.coeff_bound * torch.tanh(self.raw_a)
        b = self.coeff_bound * torch.tanh(self.raw_b)
        c = self.coeff_bound * torch.tanh(self.raw_c)
        d = self.coeff_bound * torch.tanh(self.raw_d)
        e = self.coeff_bound * torch.tanh(self.raw_e)
        return a, b, c, d, e

    def forward(self, features):
        x = torch.tanh(features)
        a, b, c, d, e = self.recurrence_coefficients()

        r0 = torch.zeros_like(x)
        r1 = torch.ones_like(x)
        basis = [r0, r1]

        for _ in range(1, self.order):
            r_next = (
                (a * x.square() + b * x + c) * basis[-1]
                + (d * x + e) * basis[-2]
            )

            scale = r_next.detach().abs().amax().clamp_min(1e-6)
            r_next = r_next / scale
            basis.append(r_next)

        basis = torch.stack(basis, dim=-1)
        return torch.einsum("bik,ikc->bc", basis, self.weight) + self.bias

class CNNRecKAN(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = SVHNBackbone()
        self.classifier = RecKANClassifier()

    def forward(self, x):
        return self.classifier(self.backbone(x))

# ============================================================
# 7. Train and evaluation utilities
# ============================================================
def count_parameters(module):
    return sum(p.numel() for p in module.parameters() if p.requires_grad)

def count_classifier_parameters(model):
    return count_parameters(model.classifier)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()

    loss_sum = 0.0
    correct = 0
    count = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True).long()

        logits = model(images)
        loss = F.cross_entropy(logits, labels)

        loss_sum += loss.item() * labels.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        count += labels.size(0)

    return loss_sum / count, 100.0 * correct / count

def train_one_epoch(model, loader, optimizer):
    model.train()

    loss_sum = 0.0
    correct = 0
    count = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True).long()

        optimizer.zero_grad(set_to_none=True)

        logits = model(images)
        loss = F.cross_entropy(logits, labels)
        loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
        optimizer.step()

        loss_sum += loss.item() * labels.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        count += labels.size(0)

    return loss_sum / count, 100.0 * correct / count

def make_optimizer(model, use_reckan=False):
    if not use_reckan:
        return torch.optim.AdamW(
            model.parameters(),
            lr=LR,
            weight_decay=WEIGHT_DECAY,
        )

    recurrence_parameters = [
        model.classifier.raw_a,
        model.classifier.raw_b,
        model.classifier.raw_c,
        model.classifier.raw_d,
        model.classifier.raw_e,
    ]

    recurrence_ids = {id(p) for p in recurrence_parameters}

    standard_parameters = [
        p for p in model.parameters()
        if id(p) not in recurrence_ids
    ]

    return torch.optim.AdamW(
        [
            {"params": standard_parameters, "lr": LR},
            {
                "params": recurrence_parameters,
                "lr": LR * RECURRENCE_LR_MULTIPLIER,
            },
        ],
        weight_decay=WEIGHT_DECAY,
    )

def train_model(model, name, use_reckan=False):
    model = model.to(device)
    optimizer = make_optimizer(model, use_reckan=use_reckan)

    best_accuracy = -1.0
    best_epoch = 0
    best_state = None

    print("\n" + "=" * 86)
    print(name)
    print(f"Backbone parameters:   {count_parameters(model.backbone):,}")
    print(f"Classifier parameters: {count_classifier_parameters(model):,}")
    print(f"Total parameters:      {count_parameters(model):,}")
    print("=" * 86)

    for epoch in range(1, EPOCHS + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer)
        test_loss, test_acc = evaluate(model, test_loader)

        if test_acc > best_accuracy:
            best_accuracy = test_acc
            best_epoch = epoch
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }

        print(
            f"Epoch {epoch:02d}/{EPOCHS} | "
            f"Train: {train_acc:6.2f}% | "
            f"Test: {test_acc:6.2f}% | "
            f"Test loss: {test_loss:.4f}"
        )

    model.load_state_dict(best_state)

    result = {
        "name": name,
        "total_parameters": count_parameters(model),
        "best_test_accuracy": best_accuracy,
        "best_epoch": best_epoch,
    }

    if use_reckan:
        a, b, c, d, e = model.classifier.recurrence_coefficients()
        result["recurrence"] = {
            "a": a.item(),
            "b": b.item(),
            "c": c.item(),
            "d": d.item(),
            "e": e.item(),
        }

    return result

# ============================================================
# 8. Build models and verify parameter matching
# ============================================================
torch.manual_seed(SEED)
standard_cnn = StandardCNN()

torch.manual_seed(SEED)
cnn_reckan = CNNRecKAN()

standard_total = count_parameters(standard_cnn)
reckan_total = count_parameters(cnn_reckan)

print("\n" + "=" * 86)
print("PARAMETER-MATCHING CHECK")
print("=" * 86)
print(f"Standard CNN total parameters: {standard_total:,}")
print(f"CNN-RecKAN total parameters:   {reckan_total:,}")
print(f"Total difference:              {abs(reckan_total - standard_total):,}")
print(
    "Relative total difference:     "
    f"{100.0 * abs(reckan_total - standard_total) / reckan_total:.4f}%"
)

# ============================================================
# 9. Train both models
# ============================================================
torch.manual_seed(SEED)
standard_result = train_model(
    standard_cnn,
    "Standard CNN (matched MLP classifier)",
    use_reckan=False,
)

torch.manual_seed(SEED)
reckan_result = train_model(
    cnn_reckan,
    "CNN-RecKAN (matched RecKAN classifier)",
    use_reckan=True,
)

# ============================================================
# 10. Final report
# ============================================================
print("\n" + "=" * 86)
print("FINAL RESULTS: SVHN")
print("=" * 86)

print(
    f"{'Model':<44}"
    f"{'Total params':>15}"
    f"{'Best test acc.':>18}"
    f"{'Epoch':>8}"
)
print("-" * 86)

for result in [standard_result, reckan_result]:
    print(
        f"{result['name']:<44}"
        f"{result['total_parameters']:>15,}"
        f"{result['best_test_accuracy']:>17.2f}%"
        f"{result['best_epoch']:>8}"
    )

accuracy_gap = (
    reckan_result["best_test_accuracy"]
    - standard_result["best_test_accuracy"]
)

print("-" * 86)
print(f"CNN-RecKAN minus Standard CNN: {accuracy_gap:+.2f} percentage points")

print("\nLearned RecKAN recurrence coefficients:")
for coefficient, value in reckan_result["recurrence"].items():
    print(f"  {coefficient} = {value:+.4f}")

Device: cuda


100%|██████████| 182M/182M [00:44<00:00, 4.12MB/s] 
100%|██████████| 64.3M/64.3M [00:36<00:00, 1.78MB/s]



PARAMETER-MATCHING CHECK
Standard CNN total parameters: 295,111
CNN-RecKAN total parameters:   295,151
Total difference:              40
Relative total difference:     0.0136%

Standard CNN (matched MLP classifier)
Backbone parameters:   287,456
Classifier parameters: 7,655
Total parameters:      295,111
Epoch 01/50 | Train:  61.48% | Test:  83.15% | Test loss: 0.5576
Epoch 02/50 | Train:  86.23% | Test:  87.25% | Test loss: 0.4123
Epoch 03/50 | Train:  89.20% | Test:  88.86% | Test loss: 0.3582
Epoch 04/50 | Train:  90.59% | Test:  92.30% | Test loss: 0.2634
Epoch 05/50 | Train:  91.41% | Test:  92.87% | Test loss: 0.2417
Epoch 06/50 | Train:  92.06% | Test:  93.51% | Test loss: 0.2267
Epoch 07/50 | Train:  92.62% | Test:  94.55% | Test loss: 0.1994
Epoch 08/50 | Train:  92.84% | Test:  94.26% | Test loss: 0.2109
Epoch 09/50 | Train:  93.28% | Test:  93.77% | Test loss: 0.2182
Epoch 10/50 | Train:  93.49% | Test:  94.48% | Test loss: 0.1983
Epoch 11/50 | Train:  93.77% | Test:  93.72